## Sistema de Recomendação

Tarefa:
1. Crie uma matriz de interação Usuário × Produto obedecendo às regras abaixo:
     - a. Linhas: id_cliente
     - b. Colunas: id_produto
     - c. Valor da célula:
     - d. 1 se o cliente comprou ao menos uma vez o produto
     - e. 0 caso contrário
     - f. Ignore a quantidade comprada (presença/ ausência apenas)


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df_vendas  = pd.read_csv('../datasets/vendas_2023_2024.csv')
df_produtos = pd.read_csv('../datasets/produtos_raw.csv').rename(columns={'code': 'id_product'})

# Matriz usuário × produto (presença/ausência)
matriz = (
    df_vendas.groupby(['id_client', 'id_product'])['id']
    .count()
    .unstack(fill_value=0)
    .clip(upper=1)   # binariza: qualquer valor > 0 vira 1
)

print(f"Matriz: {matriz.shape[0]} clientes × {matriz.shape[1]} produtos")
matriz.head()

Matriz: 49 clientes × 150 produtos


id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_client,,,,,,,,,,,,,,,,,,,,,
1,1,0,1,1,1,1,0,0,0,0,...,1,1,0,1,1,1,1,1,0,0
2,0,1,1,0,1,1,1,1,1,1,...,1,1,1,1,1,1,0,1,1,1
3,1,1,1,1,1,1,1,0,1,1,...,0,1,1,1,0,1,1,1,1,1
4,1,1,0,1,1,0,1,0,0,1,...,1,1,1,1,0,0,1,0,1,1
5,1,1,0,1,1,1,1,1,1,1,...,1,1,1,1,0,1,1,1,1,0


2. Cálculo de Similaridade entre Produtos
     - a. Calcule a Similaridade de Cosseno (Cosine Similarity) entre os vetores dos produtos
     - b. A similaridade deve ser calculada produto × produto, com base nos clientes que compraram cada item

In [3]:
# Transpõe para produto × cliente antes de calcular
sim_matrix = cosine_similarity(matriz.T)

df_sim = pd.DataFrame(
    sim_matrix,
    index=matriz.columns,
    columns=matriz.columns
)

print("Matriz de similaridade:", df_sim.shape)

Matriz de similaridade: (150, 150)


3. Ranking de Produtos Similares
     - a. Considere o produto “GPS Garmin Vortex Maré Drift” como item de referência
     - b. Gere um ranking com os 5 produtos mais similares a ele
     - c. Desconsidere o próprio GPS no ranking

In [4]:
PRODUTO_REF = 'GPS Garmin Vortex Maré Drift'

# Localiza o id do produto de referência
id_ref = df_produtos[df_produtos['name'] == PRODUTO_REF]['id_product'].values[0]
print(f"{PRODUTO_REF} → id_product: {id_ref}")

# Top 5 similares (excluindo o próprio produto)
similares = (
    df_sim[id_ref]
    .drop(index=id_ref)
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
    .rename(columns={id_ref: 'similaridade', 'id_product': 'id_product'})
)

similares.columns = ['id_product', 'similaridade']

# Adiciona nome do produto
similares = similares.merge(df_produtos[['id_product', 'name']], on='id_product')
similares['similaridade'] = similares['similaridade'].round(4)

print(f"\nTop 5 produtos recomendados junto ao '{PRODUTO_REF}':")
similares

GPS Garmin Vortex Maré Drift → id_product: 27

Top 5 produtos recomendados junto ao 'GPS Garmin Vortex Maré Drift':


,id_product,similaridade,name
0,94,0.8696,Motor de Popa Volvo Magnum 276HP
1,11,0.8680,GPS Furuno Swift Leviathan Poseidon
2,35,0.8539,Radar Furuno Swift
3,1,0.8500,Transponder AIS Maré Magnum
4,115,0.8500,Cabo de Nylon Delta Force Magnum Leviathan


### Validação

In [5]:
top1 = similares.iloc[0]
print(f"Produto com maior similaridade: id {top1['id_product']} — {top1['name']} ({top1['similaridade']})")

Produto com maior similaridade: id 94 — Motor de Popa Volvo Magnum 276HP (0.8696)
